# CARDIOVERSE: Gene Network Explanations
This sandbox demonstrates the complete explainability pipeline for identifying informative genes in cardiotoxicity prediction.

**Goal:** Load a trained gene-modality model and its validation set, generate Integrated Gradients (IG) attributions, and perform differential analysis to identify significantly relevant genes.

In [4]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import functional as F
import torch_geometric as tg
from tqdm import tqdm
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from cardioverse.models.linet import LiNetModel
from cardioverse.configs.gnn_config import GNNModelConfig
from cardioverse.explanations.integrated_gradients import IGExplainer

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Load Trained Model and Metadata
We use the bundled artifact that contains the architecture configuration, trained weights, and the specific validation sample IDs used during training.

In [5]:
def load_linet_bundle(artefact_path):
    """Reconstructs the model and metadata from the bundled artifact."""
    bundle = torch.load(artefact_path, map_location="cpu", weights_only=False)
    
    # Initialize model
    config = GNNModelConfig(**bundle["model_config"])
    model = LiNetModel(config)
    model.load_state_dict(bundle["model_state_dict"])
    model.eval()
    
    return model, config, bundle.get("val_sample_ids", [])

# Point to the latest trained gene artifact
ARTEFACT_PATH = "../artefacts/genes_linet_20260330_102049/trained_model.pt"

model, model_config, val_sample_ids = load_linet_bundle(ARTEFACT_PATH)
model = model.to(DEVICE)
print(f"Gene model and {len(val_sample_ids)} validation IDs loaded from {ARTEFACT_PATH}")

Gene model and 53 validation IDs loaded from ../artefacts/genes_linet_20260330_102049/trained_model.pt


## Load and Prepare Validation Data
We load the gene expression features and graph structure, then filter to exactly match the validation set.

In [6]:
# Load data files
X_df = pd.read_csv("../data/gene_level_data/X.csv", index_col=0)
edge_index = np.load("../data/gene_level_data/edge_index.npy")
genes = list(X_df.columns)

# Load labels
drug_metadata = pd.read_csv("../data/metadata/drug_metadata.csv")
drug_tox_map = drug_metadata.set_index("Drug name")["Is_cardiotoxic"].to_dict()

# Filter to the exact validation samples saved in the bundle
X_val_df = X_df.loc[val_sample_ids]
y_val = np.array([1 if drug_tox_map[name.split(":")[-1]] == "Yes" else 0 for name in val_sample_ids])

# Create Dataset
val_dataset = TensorDataset(
    torch.from_numpy(X_val_df.values).float(),
    torch.from_numpy(y_val).long(),
    torch.arange(len(X_val_df))
)

print(f"Validation set prepared: {len(val_dataset)} samples.")

Validation set prepared: 53 samples.


## Identify Correct Predictions
We filter for correctly classified samples to ensure high-fidelity explanations.

In [10]:
yval_true, yval_pred = [], []
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
ei_torch = torch.from_numpy(edge_index).long().to(DEVICE)

with torch.no_grad():
    for batch in val_loader:
        x, y, _ = batch
        data_list = [tg.data.Data(x=x[i].unsqueeze(1), edge_index=ei_torch) for i in range(len(x))]
        graph_batch = tg.data.Batch.from_data_list(data_list).to(DEVICE)
        logits, _ = model(graph_batch.x, graph_batch.edge_index, graph_batch.batch)
        
        yval_pred += list(F.softmax(logits, dim=1).argmax(1).cpu().numpy())
        yval_true += list(y.numpy())

val_results_df = pd.DataFrame(index=val_sample_ids, data={"ytrue": yval_true, "ypred": yval_pred})
val_results_df.sample(frac=1.).head()

,ytrue,ypred
MSN02:cabozantinib,0,0
MSN05:idarubicin,1,1
MSN01:trametinib,1,1
MSN06:bortezomib,1,1
MSN09:trametinib,1,1


##  Generate Attributions (Integrated Gradients)
Compute IG scores for all validation samples.

In [12]:
explainer = IGExplainer(model, edge_index=torch.from_numpy(edge_index).long(), device=DEVICE)

print("Generating attributions for the full validation set...")
ig_df = explainer.explain(
    val_dataset, 
    target=1,               # Attribute towards 'Toxic' class
    feature_names=genes,
    internal_batch_size=None
)
ig_df.index = val_sample_ids
print("Attributions generated.")

Generating attributions for the full validation set...


100%|███████████████████████████████████████████████████████████████████████| 53/53 [00:04<00:00, 12.14it/s]

Attributions generated.


## Differential Attribution Analysis
We perform Mann-Whitney U tests to find genes with significantly different attributions between Toxic and Non-Toxic samples.

In [13]:
# Filter to correct predictions
correct_tox_ids = val_results_df[(val_results_df.ytrue == 1) & (val_results_df.ytrue == val_results_df.ypred)].index
correct_notox_ids = val_results_df[(val_results_df.ytrue == 0) & (val_results_df.ytrue == val_results_df.ypred)].index

ig_tox = ig_df.loc[correct_tox_ids]
ig_notox = ig_df.loc[correct_notox_ids]

print(f"Analyzing {len(ig_tox)} True Positives vs {len(ig_notox)} True Negatives...")

pvals = []
for gene in tqdm(ig_df.columns):
    _, p = mannwhitneyu(ig_tox[gene], ig_notox[gene], alternative='two-sided')
    pvals.append(p)

# Correct for multiple hypothesis testing (FDR BH)
reject, pvals_adj, _, _ = multipletests(pvals, method='fdr_bh')

# Compile full results
infr_df = pd.DataFrame({
    'pval': pvals,
    'pval_adj': pvals_adj,
    '-log10padj': -np.log10(pvals_adj),
    'reject': reject,
    'mean_Tox': ig_tox.mean(),
    'mean_NoTox': ig_notox.mean(),
    'mean_diff': ig_tox.mean() - ig_notox.mean()
}, index=genes).sort_values('pval_adj')

Analyzing 19 True Positives vs 26 True Negatives...


100%|███████████████████████████████████████████████████████████████████| 916/916 [00:00<00:00, 1672.47it/s]


## Top 20 Significant Genes
Identify the top 20 genes most significantly associated with the model's toxicity predictions.

In [15]:
pd.options.display.float_format = "{:.2e}".format
print(" Top 20 Most Informative Genes:")
infr_df.head(20)

 Top 20 Most Informative Genes:


,pval,pval_adj,-log10padj,reject,mean_Tox,mean_NoTox,mean_diff
HSPA4,2.86e-08,8.52e-06,5.07e+00,True,-2.21e-03,-4.54e-06,-2.21e-03
PRKAB2,1.93e-08,8.52e-06,5.07e+00,True,-5.94e-03,-2.28e-06,-5.94e-03
SLC8A1,3.72e-08,8.52e-06,5.07e+00,True,1.43e-02,1.40e-05,1.43e-02
INPPL1,2.51e-08,8.52e-06,5.07e+00,True,-3.07e-03,1.29e-06,-3.07e-03
PRKAG2,5.14e-07,4.28e-05,4.37e+00,True,3.04e-03,5.82e-06,3.03e-03
HMOX1,4.04e-07,4.28e-05,4.37e+00,True,2.43e-02,-2.19e-06,2.43e-02
HLA-E,4.04e-07,4.28e-05,4.37e+00,True,4.12e-02,-3.35e-06,4.12e-02
GRK6,5.14e-07,4.28e-05,4.37e+00,True,7.25e-03,-2.23e-06,7.26e-03
DDX5,3.58e-07,4.28e-05,4.37e+00,True,2.05e-03,-1.63e-05,2.07e-03
FYN,2.81e-07,4.28e-05,4.37e+00,True,4.53e-03,-1.15e-05,4.54e-03
